# Week 6 Isolation Forest Modelling — huaye

## 1. Objective

This notebook applies Isolation Forest to the shared Week 6 ML-ready Floor 6 dataset. It uses exactly the feature contract used by Gavin's LOF analysis, evaluates contamination settings 0.03, 0.05, and 0.09, and prepares descriptive comparisons with the v3 statistical baseline and LOF.

In [1]:
from __future__ import annotations

import json
import os
import time
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
ML_READY_DIR = PROJECT_DIR / "outputs" / "floor6_zone1245_ml_ready"
ML_READY_PATH = ML_READY_DIR / "floor6_zone1245_ml_ready_2018_jul01_sep01.csv.gz"
FEATURE_LIST_PATH = ML_READY_DIR / "feature_list.json"
LOF_RESULTS_PATH = (
    PROJECT_DIR
    / "outputs"
    / "floor6_zone1245_lof_modelling_gavin"
    / "lof_anomaly_results_full.csv.gz"
)
OUTPUT_DIR = PROJECT_DIR / "outputs" / "isolation_forest_huaye"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AUTHOR = "huaye"
EXPECTED_ROWS = 362_864
EXPECTED_FLOOR = 6
EXPECTED_ZONES = [1, 2, 4, 5]
EXPECTED_FEATURE_COUNT = 42
KEY_COLUMNS = ["timestamp", "floor", "zone"]
REFERENCE_COLUMNS = [
    "timestamp",
    "floor",
    "zone",
    "hour",
    "operating_period",
    "baseline_anomaly_label",
    "baseline_anomaly_count",
]
CONTAMINATION_SETTINGS = [0.03, 0.05, 0.09]
PRIMARY_CONTAMINATION = 0.05
FIXED_PARAMETERS = {
    "n_estimators": 100,
    "max_samples": "auto",
    "max_features": 1.0,
    "bootstrap": False,
    "random_state": 42,
    "n_jobs": -1,
}

print(f"Project directory: {PROJECT_DIR}")
print(f"Shared ML-ready dataset: {ML_READY_PATH}")
print(f"Huaye output directory: {OUTPUT_DIR}")

Project directory: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis
Shared ML-ready dataset: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\floor6_zone1245_ml_ready\floor6_zone1245_ml_ready_2018_jul01_sep01.csv.gz
Huaye output directory: D:\实习任务\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye


## 2. Shared ML-Ready Dataset

The input is Lee's shared ML-ready dataset. Identifiers and statistical-baseline fields are retained only as references.

In [2]:
if not ML_READY_PATH.exists():
    raise FileNotFoundError(f"Missing shared ML-ready dataset: {ML_READY_PATH}")
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(f"Missing feature manifest: {FEATURE_LIST_PATH}")

feature_manifest = json.loads(FEATURE_LIST_PATH.read_text(encoding="utf-8"))
model_features = feature_manifest["model_feature_columns"]
required_columns = REFERENCE_COLUMNS + model_features
ml_ready = pd.read_csv(ML_READY_PATH, usecols=required_columns, parse_dates=["timestamp"])

missing_columns = sorted(set(required_columns) - set(ml_ready.columns))
if missing_columns:
    raise ValueError(f"ML-ready dataset is missing columns: {missing_columns}")
if len(ml_ready) != EXPECTED_ROWS:
    raise ValueError(f"Expected {EXPECTED_ROWS:,} rows, found {len(ml_ready):,}")
if sorted(ml_ready["floor"].unique().tolist()) != [EXPECTED_FLOOR]:
    raise ValueError(f"Unexpected floors: {sorted(ml_ready['floor'].unique().tolist())}")
if sorted(ml_ready["zone"].unique().tolist()) != EXPECTED_ZONES:
    raise ValueError(f"Unexpected zones: {sorted(ml_ready['zone'].unique().tolist())}")
if ml_ready.duplicated(KEY_COLUMNS).any():
    raise ValueError("Duplicate timestamp/floor/zone keys found")

print(f"Rows loaded: {len(ml_ready):,}")
print(f"Timestamp range: {ml_ready['timestamp'].min()} to {ml_ready['timestamp'].max()}")
print(f"Zones: {sorted(ml_ready['zone'].unique().tolist())}")
ml_ready[REFERENCE_COLUMNS].head()

Rows loaded: 362,864
Timestamp range: 2018-07-01 00:04:00 to 2018-09-01 23:59:00
Zones: [1, 2, 4, 5]


,timestamp,floor,zone,hour,operating_period,baseline_anomaly_label,baseline_anomaly_count
0,2018-07-01 00:04:00,6,1,0,weekend,0,0
1,2018-07-01 00:05:00,6,1,0,weekend,0,0
2,2018-07-01 00:06:00,6,1,0,weekend,0,0
3,2018-07-01 00:07:00,6,1,0,weekend,0,0
4,2018-07-01 00:08:00,6,1,0,weekend,0,0


## 3. Feature Matrix

The feature matrix is selected directly from `feature_list.json`. No additional scaling or encoding is applied because Lee's pipeline has already completed those transformations.

In [3]:
for forbidden in [
    "timestamp",
    "floor",
    "baseline_anomaly_label",
    "baseline_anomaly_count",
]:
    if forbidden in model_features:
        raise AssertionError(f"Forbidden model feature detected: {forbidden}")
if len(model_features) != EXPECTED_FEATURE_COUNT:
    raise ValueError(f"Expected {EXPECTED_FEATURE_COUNT} features, found {len(model_features)}")
if ml_ready[model_features].isna().any().any():
    raise ValueError("Model features contain missing values")

X = ml_ready[model_features].to_numpy(dtype=np.float32)
if not np.isfinite(X).all():
    raise ValueError("Model features contain infinite values")

print(f"Feature matrix shape: {X.shape}")
print("No additional scaling or encoding was applied.")
pd.DataFrame({"model_feature": model_features})

Feature matrix shape: (362864, 42)
No additional scaling or encoding was applied.


,model_feature
0,numerical__total_AC_power_kw_rolling_mean_15min
1,numerical__light_power_kw_rolling_mean_15min
2,numerical__plug_power_kw_rolling_mean_15min
3,numerical__temperature_c_rolling_mean_15min
4,numerical__humidity_pct_rolling_mean_15min
5,numerical__lux_rolling_mean_15min
6,numerical__minute_of_day_sin
7,numerical__minute_of_day_cos
8,categorical__zone_1
9,categorical__zone_2


## 4. Isolation Forest Method

`IsolationForest.score_samples(X)` returns lower values for observations considered more abnormal. For interpretation, this notebook defines `if_anomaly_score = -model.score_samples(X)`, so a higher user-facing score means a more anomalous observation. Labels use the fitted model's contamination-dependent offset, with `1` for anomaly and `0` for normal.

In [4]:
def contamination_suffix(contamination: float) -> str:
    return f"c{int(round(contamination * 1000)):03d}"


results = ml_ready[REFERENCE_COLUMNS].copy()
model_records = []
primary_score = None

for contamination in CONTAMINATION_SETTINGS:
    start_time = time.time()
    model = IsolationForest(contamination=contamination, **FIXED_PARAMETERS)
    model.fit(X)
    raw_score = model.score_samples(X)
    anomaly_score = -raw_score
    labels = (raw_score < model.offset_).astype(np.int8)
    suffix = contamination_suffix(contamination)
    label_column = f"if_label_{suffix}"
    results[label_column] = labels
    if contamination == PRIMARY_CONTAMINATION:
        primary_score = anomaly_score
    model_records.append(
        {
            "contamination": contamination,
            "anomaly_count": int(labels.sum()),
            "anomaly_percentage": float(100 * labels.mean()),
            "fit_and_score_seconds": float(time.time() - start_time),
            "offset": float(model.offset_),
        }
    )

if primary_score is None:
    raise AssertionError("Primary contamination score was not generated")
results["if_anomaly_score"] = primary_score
results["if_anomaly_label"] = results["if_label_c050"].astype(np.int8)
parameter_comparison = pd.DataFrame(model_records)
parameter_comparison

,contamination,anomaly_count,anomaly_percentage,fit_and_score_seconds,offset
0,0.03,10882,2.998920,2.494246,-0.478227
1,0.05,18143,4.999945,2.361010,-0.470456
2,0.09,32658,9.000066,2.438705,-0.462212


## 5. Model Parameters

The three contamination values are sensitivity settings. The 0.05 setting is primary only to support direct comparison with Gavin's LOF result; it is not claimed to be objectively optimal.

In [5]:
display(pd.DataFrame([FIXED_PARAMETERS]))
display(parameter_comparison)

,n_estimators,max_samples,max_features,bootstrap,random_state,n_jobs
0,100,auto,1.0,False,42,-1


,contamination,anomaly_count,anomaly_percentage,fit_and_score_seconds,offset
0,0.03,10882,2.998920,2.494246,-0.478227
1,0.05,18143,4.999945,2.361010,-0.470456
2,0.09,32658,9.000066,2.438705,-0.462212


## 6. Contamination Sensitivity Analysis

Jaccard indices below measure overlap between anomaly sets only; they are not measures of model accuracy.

In [6]:
overlap_records = []
for setting_a, setting_b in combinations(CONTAMINATION_SETTINGS, 2):
    column_a = f"if_label_{contamination_suffix(setting_a)}"
    column_b = f"if_label_{contamination_suffix(setting_b)}"
    anomaly_a = results[column_a].eq(1)
    anomaly_b = results[column_b].eq(1)
    intersection = int((anomaly_a & anomaly_b).sum())
    union = int((anomaly_a | anomaly_b).sum())
    overlap_records.append(
        {
            "setting_a": setting_a,
            "setting_b": setting_b,
            "intersection": intersection,
            "union": union,
            "jaccard": intersection / union if union else np.nan,
        }
    )
parameter_overlap = pd.DataFrame(overlap_records)
parameter_overlap

,setting_a,setting_b,intersection,union,jaccard
0,0.03,0.05,10882,18143,0.599791
1,0.03,0.09,10882,32658,0.333211
2,0.05,0.09,18143,32658,0.555545


## 7. Primary 0.05 Comparison Setting

The primary Isolation Forest label is the 0.05 contamination result. Statistical-baseline labels remain external comparison references.

In [7]:
if_mask = results["if_anomaly_label"].eq(1)
baseline_mask = results["baseline_anomaly_label"].eq(1)
if_count = int(if_mask.sum())
baseline_count = int(baseline_mask.sum())
if_baseline_overlap = int((if_mask & baseline_mask).sum())
if_only = int((if_mask & ~baseline_mask).sum())
baseline_only = int((~if_mask & baseline_mask).sum())
if_baseline_union = int((if_mask | baseline_mask).sum())
if_baseline_jaccard = if_baseline_overlap / if_baseline_union if if_baseline_union else np.nan

baseline_comparison = pd.DataFrame(
    [
        {
            "if_contamination": PRIMARY_CONTAMINATION,
            "if_anomaly_count": if_count,
            "baseline_anomaly_count": baseline_count,
            "overlap_count": if_baseline_overlap,
            "if_only_count": if_only,
            "baseline_only_count": baseline_only,
            "union_count": if_baseline_union,
            "jaccard_overlap": if_baseline_jaccard,
        }
    ]
)
baseline_comparison

,if_contamination,if_anomaly_count,baseline_anomaly_count,overlap_count,if_only_count,baseline_only_count,union_count,jaccard_overlap
0,0.05,18143,32777,2744,15399,30033,48176,0.056958


## 8. Anomaly Distribution by Zone

In [8]:
def summarise_distribution(frame: pd.DataFrame, group_column: str) -> pd.DataFrame:
    grouped = frame.groupby(group_column, dropna=False)
    summary = grouped.agg(
        rows=("if_anomaly_label", "size"),
        if_anomalies_c030=("if_label_c030", "sum"),
        if_anomalies_c050=("if_label_c050", "sum"),
        if_anomalies_c090=("if_label_c090", "sum"),
        baseline_anomalies=("baseline_anomaly_label", "sum"),
        mean_if_anomaly_score=("if_anomaly_score", "mean"),
    ).reset_index()
    for suffix in ["c030", "c050", "c090"]:
        summary[f"if_percentage_{suffix}"] = (
            100 * summary[f"if_anomalies_{suffix}"] / summary["rows"]
        )
    return summary.sort_values(group_column, kind="stable").reset_index(drop=True)

zone_summary = summarise_distribution(results, "zone")
zone_summary

,zone,rows,if_anomalies_c030,if_anomalies_c050,if_anomalies_c090,baseline_anomalies,mean_if_anomaly_score,if_percentage_c030,if_percentage_c050,if_percentage_c090
0,1,90716,2755,5486,10196,12927,0.431741,3.036950,6.047445,11.239473
1,2,90716,2705,4355,7568,5475,0.430074,2.981833,4.800697,8.342520
2,4,90716,3338,5300,10091,5861,0.429701,3.679616,5.842409,11.123727
3,5,90716,2084,3002,4803,8514,0.422651,2.297279,3.309229,5.294546


## 9. Anomaly Distribution by Hour

In [9]:
hour_summary = summarise_distribution(results, "hour")
hour_summary.sort_values("if_percentage_c050", ascending=False).head(10)

,hour,rows,if_anomalies_c030,if_anomalies_c050,if_anomalies_c090,baseline_anomalies,mean_if_anomaly_score,if_percentage_c030,if_percentage_c050,if_percentage_c090
7,7,15120,8704,10179,10800,1065,0.467654,57.566138,67.321429,71.428571
17,17,15120,1535,4330,8359,1768,0.458846,10.152116,28.637566,55.284392
10,10,15120,376,1603,3660,2001,0.447236,2.486772,10.601852,24.206349
18,18,15120,116,673,3385,2208,0.450179,0.767196,4.451058,22.387566
14,14,15120,40,389,1908,2422,0.439051,0.264550,2.572751,12.619048
8,8,15120,69,381,1242,2157,0.438657,0.456349,2.519841,8.214286
9,9,15120,42,298,1837,2117,0.437630,0.277778,1.970899,12.149471
16,16,15120,0,217,1063,1800,0.438154,0.000000,1.435185,7.030423
11,11,15120,0,30,122,1815,0.431648,0.000000,0.198413,0.806878
13,13,15120,0,23,81,1823,0.432833,0.000000,0.152116,0.535714


## 10. Anomaly Distribution by Operating Period

In [10]:
operating_period_summary = summarise_distribution(results, "operating_period")
operating_period_summary.sort_values("if_percentage_c050", ascending=False)

,operating_period,rows,if_anomalies_c030,if_anomalies_c050,if_anomalies_c090,baseline_anomalies,mean_if_anomaly_score,if_percentage_c030,if_percentage_c050,if_percentage_c090
4,ramp_up,10800,8704,10179,10800,1065,0.490071,80.592593,94.250000,100.000000
3,ramp_down,21600,1651,4992,11698,3556,0.464219,7.643519,23.111111,54.157407
0,core_work,97200,527,2952,9950,16245,0.442657,0.542181,3.037037,10.236626
2,night_off,75600,0,9,97,5875,0.417373,0.000000,0.011905,0.128307
5,weekend,103664,0,11,107,3269,0.414618,0.000000,0.010611,0.103218
1,evening_off,54000,0,0,6,2767,0.418924,0.000000,0.000000,0.011111


## 11. Statistical Baseline Comparison

At contamination 0.05, the statistical baseline identifies 32,777 anomalies. Its overlap with Isolation Forest is 2,744 observations (Jaccard index 0.056958). This is a descriptive comparison rather than an accuracy evaluation: the statistical baseline is a comparison reference, not verified ground truth.

In [11]:
baseline_comparison

,if_contamination,if_anomaly_count,baseline_anomaly_count,overlap_count,if_only_count,baseline_only_count,union_count,jaccard_overlap
0,0.05,18143,32777,2744,15399,30033,48176,0.056958


## 12. Isolation Forest vs LOF Comparison

At contamination 0.05, Isolation Forest and LOF each identify 18,143 anomalies from the same rows and the same 42-feature representation. Their overlap is 1,049 observations (Jaccard index 0.029770). The low overlap shows that the two unsupervised methods identify substantially different subsets of unusual observations; it does not establish that either model is better. This difference is treated as an evaluation finding concerning method sensitivity and agreement.

In [12]:
if not LOF_RESULTS_PATH.exists():
    raise FileNotFoundError(f"Missing Gavin LOF result: {LOF_RESULTS_PATH}")

lof_results = pd.read_csv(
    LOF_RESULTS_PATH,
    usecols=KEY_COLUMNS + ["lof_anomaly_label"],
    parse_dates=["timestamp"],
)
if lof_results.duplicated(KEY_COLUMNS).any():
    raise ValueError("Gavin LOF result contains duplicate row keys")

aligned = results[KEY_COLUMNS + ["if_anomaly_label"]].merge(
    lof_results, on=KEY_COLUMNS, how="outer", validate="one_to_one", indicator=True
)
if len(aligned) != len(results) or not aligned["_merge"].eq("both").all():
    raise ValueError("IF and LOF row keys do not align exactly")

aligned_if = aligned["if_anomaly_label"].eq(1)
aligned_lof = aligned["lof_anomaly_label"].eq(1)
if_lof_overlap = int((aligned_if & aligned_lof).sum())
if_lof_union = int((aligned_if | aligned_lof).sum())
if_lof_comparison = pd.DataFrame(
    [
        {
            "contamination": PRIMARY_CONTAMINATION,
            "rows_aligned": len(aligned),
            "if_anomaly_count": int(aligned_if.sum()),
            "lof_anomaly_count": int(aligned_lof.sum()),
            "overlap_count": if_lof_overlap,
            "if_only_count": int((aligned_if & ~aligned_lof).sum()),
            "lof_only_count": int((~aligned_if & aligned_lof).sum()),
            "union_count": if_lof_union,
            "jaccard_overlap": if_lof_overlap / if_lof_union if if_lof_union else np.nan,
        }
    ]
)
if_lof_comparison

,contamination,rows_aligned,if_anomaly_count,lof_anomaly_count,overlap_count,if_only_count,lof_only_count,union_count,jaccard_overlap
0,0.05,362864,18143,18143,1049,17094,17094,35237,0.02977


## 13. Limitations

- Isolation Forest is an unsupervised anomaly detection method.
- Statistical baseline labels are used only for comparison and are not ground truth.
- Detected anomalies represent unusual observations, not confirmed equipment faults.
- The contamination parameter controls the proportion of observations flagged as anomalies.
- Results apply to the current Floor 6 CU-BEMS subset and preprocessing configuration.
- Model comparison does not by itself establish which method is operationally superior.
- The results are sensitive to temporal and operating-context features. Of the 42 inputs, 6 are physical rolling sensor features, 2 are cyclic time features, 4 are zone one-hot features, 24 are hour one-hot features, and 6 are operating-period one-hot features. Contextual variables therefore form a large proportion of the representation.
- At contamination 0.05, 10,179 of 10,800 weekday ramp-up observations are flagged (94.25%). This concentration was verified and is not attributable to row-alignment or operating-period assignment errors. Ramp-up observations have a systematically higher Isolation Forest score distribution, and the top-ranked anomalies are concentrated at weekday hour 7. However, many top anomalies also exhibit unusual AC and plug-power values, so the result cannot be attributed solely to categorical or contextual features. It must not be interpreted as a confirmed equipment fault.
- Future sensitivity analysis could examine physical-only modelling, reduce duplicated temporal/context representations, fit period-specific anomaly models, and conduct feature-ablation experiments. These extensions are not required for the current Week 6 deliverable.

## 14. Generated Outputs

In [13]:
result_columns = REFERENCE_COLUMNS + [
    "if_label_c030",
    "if_label_c050",
    "if_label_c090",
    "if_anomaly_score",
    "if_anomaly_label",
]
full_results = results[result_columns].copy()
top_anomalies = (
    full_results.sort_values("if_anomaly_score", ascending=False, kind="stable")
    .head(100)
    .reset_index(drop=True)
)

output_paths = {
    "full_results": OUTPUT_DIR / "if_anomaly_results_full.csv.gz",
    "parameter_comparison": OUTPUT_DIR / "if_parameter_comparison.csv",
    "parameter_overlap": OUTPUT_DIR / "if_parameter_overlap.csv",
    "zone_summary": OUTPUT_DIR / "if_summary_by_zone.csv",
    "hour_summary": OUTPUT_DIR / "if_summary_by_hour.csv",
    "operating_period_summary": OUTPUT_DIR / "if_summary_by_operating_period.csv",
    "baseline_comparison": OUTPUT_DIR / "if_baseline_comparison.csv",
    "top_anomalies": OUTPUT_DIR / "if_top_anomalies.csv",
    "if_lof_comparison": OUTPUT_DIR / "if_lof_comparison.csv",
    "modelling_summary": OUTPUT_DIR / "if_modelling_summary.json",
}

full_results.to_csv(output_paths["full_results"], index=False)
parameter_comparison[["contamination", "anomaly_count", "anomaly_percentage"]].to_csv(
    output_paths["parameter_comparison"], index=False
)
parameter_overlap.to_csv(output_paths["parameter_overlap"], index=False)
zone_summary.to_csv(output_paths["zone_summary"], index=False)
hour_summary.to_csv(output_paths["hour_summary"], index=False)
operating_period_summary.to_csv(output_paths["operating_period_summary"], index=False)
baseline_comparison.to_csv(output_paths["baseline_comparison"], index=False)
top_anomalies.to_csv(output_paths["top_anomalies"], index=False)
if_lof_comparison.to_csv(output_paths["if_lof_comparison"], index=False)

source_keys = ml_ready[KEY_COLUMNS].reset_index(drop=True)
result_keys = full_results[KEY_COLUMNS].reset_index(drop=True)
label_columns = ["if_label_c030", "if_label_c050", "if_label_c090", "if_anomaly_label"]
rate_by_contamination = {
    row.contamination: row.anomaly_percentage / 100
    for row in parameter_comparison.itertuples(index=False)
}
validation_records = [
    {"check": "full_result_rows", "passed": len(full_results) == EXPECTED_ROWS, "evidence": f"rows={len(full_results):,}"},
    {"check": "unique_row_key", "passed": not full_results.duplicated(KEY_COLUMNS).any(), "evidence": f"duplicates={int(full_results.duplicated(KEY_COLUMNS).sum())}"},
    {"check": "score_complete", "passed": not full_results["if_anomaly_score"].isna().any(), "evidence": f"missing={int(full_results['if_anomaly_score'].isna().sum())}"},
    {"check": "binary_labels", "passed": all(set(full_results[c].unique()).issubset({0, 1}) for c in label_columns), "evidence": str({c: sorted(full_results[c].unique().tolist()) for c in label_columns})},
    {"check": "contamination_003", "passed": abs(rate_by_contamination[0.03] - 0.03) < 0.001, "evidence": f"rate={rate_by_contamination[0.03]:.6f}"},
    {"check": "contamination_005", "passed": abs(rate_by_contamination[0.05] - 0.05) < 0.001, "evidence": f"rate={rate_by_contamination[0.05]:.6f}"},
    {"check": "contamination_009", "passed": abs(rate_by_contamination[0.09] - 0.09) < 0.001, "evidence": f"rate={rate_by_contamination[0.09]:.6f}"},
    {"check": "expected_zones", "passed": sorted(full_results["zone"].unique().tolist()) == EXPECTED_ZONES, "evidence": str(sorted(full_results['zone'].unique().tolist()))},
    {"check": "feature_count", "passed": X.shape[1] == EXPECTED_FEATURE_COUNT, "evidence": f"features={X.shape[1]}"},
    {"check": "baseline_label_excluded", "passed": "baseline_anomaly_label" not in model_features, "evidence": "comparison reference only"},
    {"check": "baseline_count_excluded", "passed": "baseline_anomaly_count" not in model_features, "evidence": "comparison reference only"},
    {"check": "no_additional_preprocessing", "passed": True, "evidence": "X selected directly from model_feature_columns"},
    {"check": "source_keys_preserved", "passed": source_keys.equals(result_keys), "evidence": f"keys={len(result_keys):,}"},
]

limitations = [
    "Isolation Forest is an unsupervised anomaly detection method.",
    "Statistical baseline labels are used only for comparison and are not ground truth.",
    "Detected anomalies represent unusual observations, not confirmed equipment faults.",
    "The contamination parameter controls the proportion of observations flagged as anomalies.",
    "Results apply to the current Floor 6 CU-BEMS subset and preprocessing configuration.",
    "Model comparison does not by itself establish which method is operationally superior.",
]
summary_payload = {
    "author": AUTHOR,
    "source_dataset": str(ML_READY_PATH.relative_to(PROJECT_DIR)),
    "feature_manifest": str(FEATURE_LIST_PATH.relative_to(PROJECT_DIR)),
    "row_count": int(len(full_results)),
    "feature_count": int(len(model_features)),
    "model": "sklearn.ensemble.IsolationForest",
    "fixed_parameters": FIXED_PARAMETERS,
    "contamination_settings": CONTAMINATION_SETTINGS,
    "primary_comparison_contamination": PRIMARY_CONTAMINATION,
    "anomaly_counts": {f"{row.contamination:.2f}": int(row.anomaly_count) for row in parameter_comparison.itertuples(index=False)},
    "output_files": {name: str(path.relative_to(PROJECT_DIR)) for name, path in output_paths.items()},
    "score_direction_definition": "if_anomaly_score = -IsolationForest.score_samples(X); higher values are more anomalous",
    "random_state": FIXED_PARAMETERS["random_state"],
    "baseline_comparison": baseline_comparison.iloc[0].to_dict(),
    "lof_comparison": if_lof_comparison.iloc[0].to_dict(),
    "limitations": limitations,
    "validation": validation_records,
}
output_paths["modelling_summary"].write_text(
    json.dumps(summary_payload, indent=2, default=lambda value: value.item() if hasattr(value, "item") else str(value)),
    encoding="utf-8",
)

all_outputs_exist = all(path.exists() for path in output_paths.values())
validation_records.append(
    {"check": "all_expected_outputs", "passed": all_outputs_exist, "evidence": f"files={len(output_paths)}"}
)
validation_records.append(
    {"check": "notebook_reached_final_output_stage", "passed": True, "evidence": "final output cell executed"}
)
summary_payload["validation"] = validation_records
output_paths["modelling_summary"].write_text(
    json.dumps(summary_payload, indent=2, default=lambda value: value.item() if hasattr(value, "item") else str(value)),
    encoding="utf-8",
)

validation = pd.DataFrame(validation_records)
if not validation["passed"].all():
    raise AssertionError("One or more Huaye Isolation Forest validations failed")

print("Generated Huaye files:")
for path in output_paths.values():
    print(f"- {path.relative_to(PROJECT_DIR)}")
validation

Generated Huaye files:
- outputs\isolation_forest_huaye\if_anomaly_results_full.csv.gz
- outputs\isolation_forest_huaye\if_parameter_comparison.csv
- outputs\isolation_forest_huaye\if_parameter_overlap.csv
- outputs\isolation_forest_huaye\if_summary_by_zone.csv
- outputs\isolation_forest_huaye\if_summary_by_hour.csv
- outputs\isolation_forest_huaye\if_summary_by_operating_period.csv
- outputs\isolation_forest_huaye\if_baseline_comparison.csv
- outputs\isolation_forest_huaye\if_top_anomalies.csv
- outputs\isolation_forest_huaye\if_lof_comparison.csv
- outputs\isolation_forest_huaye\if_modelling_summary.json


,check,passed,evidence
0,full_result_rows,True,"rows=362,864"
1,unique_row_key,True,duplicates=0
2,score_complete,True,missing=0
3,binary_labels,True,"{'if_label_c030': [0, 1], 'if_label_c050': [0,..."
4,contamination_003,True,rate=0.029989
5,contamination_005,True,rate=0.049999
6,contamination_009,True,rate=0.090001
7,expected_zones,True,"[1, 2, 4, 5]"
8,feature_count,True,features=42
9,baseline_label_excluded,True,comparison reference only


## 15. Summary

The complete shared dataset was modelled with the same 42 features used by LOF. At the primary contamination setting of 0.05, Isolation Forest flags 18,143 observations. The pronounced weekday ramp-up concentration is internally consistent and reflects sensitivity to both operating context and unusual physical measurements. Agreement with LOF and the statistical baseline is low, demonstrating method sensitivity rather than model superiority. These exploratory anomaly labels identify unusual observations, not confirmed causal or equipment faults.

In [14]:
print("Huaye Isolation Forest modelling completed successfully.")
print(f"Rows modelled: {len(full_results):,}")
print(f"Features used: {len(model_features)}")
print(f"Primary 0.05 IF anomalies: {if_count:,}")
print(f"IF/statistical-baseline overlap: {if_baseline_overlap:,}")
print(f"IF/LOF overlap: {if_lof_overlap:,}")
print("All validations passed.")

Huaye Isolation Forest modelling completed successfully.
Rows modelled: 362,864
Features used: 42
Primary 0.05 IF anomalies: 18,143
IF/statistical-baseline overlap: 2,744
IF/LOF overlap: 1,049
All validations passed.
